# 🖊️ Task 1: Handwritten Text Generation
## Character-Level RNN (LSTM) – CodSoft AI/ML Internship

## 💾 Step 1: Mount Google Drive & Setup Directories

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/CodSoft_Task1'
MODELS_DIR = os.path.join(SAVE_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
print(f'✅ Models will be saved to: {MODELS_DIR}')

## 🔑 Step 2: Kaggle Credentials & Download Image Dataset

In [ ]:
import os
from getpass import getpass

print('🔑 Please enter your Kaggle API Token:')
token = getpass('Token: ')
os.environ['KAGGLE_API_TOKEN'] = token

DATA_DIR = '/content/dataset'
CSV_PATH = os.path.join(DATA_DIR, 'english.csv')

if not os.path.exists(CSV_PATH):
    print('📥 Downloading dataset...')
    !pip install kaggle -q
    os.makedirs(DATA_DIR, exist_ok=True)
    !kaggle datasets download -d dhruvildave/english-handwritten-characters-dataset -p {DATA_DIR} --unzip
    print('✅ Image Dataset downloaded!')
else:
    print('✅ Image Dataset already exists.')

## 📦 Step 3: Install Dependencies & Import Libraries

In [ ]:
!pip install tensorflow numpy pandas matplotlib Pillow opencv-python tqdm gradio -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import random
import urllib.request
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('✅ GPU Active!')
else:
    print('❌ No GPU found.')

## 🖼️ Step 4: Build Image Bank (For Rendering)

In [ ]:
df = pd.read_csv(CSV_PATH)
if len(df.columns) == 2:
    df.columns = ['image', 'label']

def load_image(img_filename, size=(64, 64)):
    path = os.path.join(DATA_DIR, img_filename) # img_filename already contains 'Img/'
    if not os.path.exists(path): return None
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    return cv2.resize(img, size)

image_bank = {}
for _, row in tqdm(df.iterrows(), total=len(df), desc='Building image bank'):
    char = str(row['label'])
    fname = row['image']
    if char not in image_bank: image_bank[char] = []
    image_bank[char].append(fname)
print(f'✅ Image bank ready with {len(image_bank)} characters')

## 📖 Step 5: Download Real Text Corpus for RNN Training

In [ ]:
print('Downloading Shakespeare corpus so the AI learns real English words...')
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

with open("shakespeare.txt", "r") as f:
    raw_text = f.read()[:300000] # Use 300k chars for training

# Filter corpus to ONLY include characters we have images for, plus spaces
valid_chars = set(image_bank.keys())
valid_chars.add(' ')
corpus = ''.join([c for c in raw_text if c in valid_chars])

vocab = sorted(set(corpus))
vocab_size = len(vocab)
char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = {i: c for c, i in char2idx.items()}
corpus_encoded = [char2idx[c] for c in corpus]

print(f'Filtered Corpus length: {len(corpus)}')
print(f'Vocabulary size: {vocab_size} {vocab}')

## 🔢 Step 6: Create Training Sequences

In [ ]:
SEQ_LENGTH = 40
STEP = 3
BATCH_SIZE = 512

X_seqs, y_seqs = [], []
for i in range(0, len(corpus_encoded) - SEQ_LENGTH, STEP):
    X_seqs.append(corpus_encoded[i : i + SEQ_LENGTH])
    y_seqs.append(corpus_encoded[i + SEQ_LENGTH])

X_train = np.array(X_seqs)
y_train = to_categorical(np.array(y_seqs), num_classes=vocab_size)
print(f'Train shapes: X={X_train.shape}, y={y_train.shape}')

## 🧠 Step 7: Train the LSTM Model

In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=SEQ_LENGTH),
    LSTM(256, return_sequences=True),
    Dropout(0.2),
    LSTM(256),
    Dense(vocab_size, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
best_model_path = os.path.join(MODELS_DIR, 'char_rnn_best.h5')
callbacks = [ModelCheckpoint(best_model_path, monitor='loss', save_best_only=True)]

print('Training model (This will take a few minutes)...')
history = model.fit(X_train, y_train, batch_size=BATCH_SIZE, epochs=15, callbacks=callbacks)
print('✅ Model Trained!')

## ✨ Step 8: Text Generation & Handwriting Rendering Logic

In [ ]:
def generate_text(model, seed_chars, min_generate=50, temperature=1.0):
    seed = [c for c in seed_chars if c in char2idx]
    if len(seed) < SEQ_LENGTH:
        seed = [char2idx[' ']] * (SEQ_LENGTH - len(seed)) + [char2idx[c] for c in seed]
    else:
        seed = [char2idx[c] for c in seed[-SEQ_LENGTH:]]

    generated = []
    current_seq = seed
    count = 0

    while True:
        x = np.array([current_seq])
        preds = model.predict(x, verbose=0)[0]
        preds = np.asarray(preds).astype('float64')
        preds = np.log(preds + 1e-8) / temperature
        exp_preds = np.exp(preds - np.max(preds))
        preds = exp_preds / np.sum(exp_preds)
        next_idx = np.argmax(np.random.multinomial(1, preds, 1))
        next_char = idx2char[next_idx]
        
        generated.append(next_char)
        current_seq = current_seq[1:] + [next_idx]
        count += 1
        
        # Stop condition: Reached minimum length AND finished the word (hit a space)
        if count >= min_generate and next_char == ' ':
            break
        # Safety limit to prevent infinite loops if it forgets spaces
        if count >= min_generate + 25:
            break

    return generated

def render_handwritten(char_sequence, char_size=64, padding=4):
    char_images = []
    for char in char_sequence:
        if char in image_bank and image_bank[char]:
            img = load_image(random.choice(image_bank[char]), size=(char_size, char_size))
            if img is not None:
                char_images.append(img)
                continue
        # Fallback for spaces or missing chars (White block)
        char_images.append(np.full((char_size, char_size), 255, dtype=np.uint8))
    if not char_images: return np.full((64, 64), 255, dtype=np.uint8)
    pad = np.full((char_size, padding), 255, dtype=np.uint8)
    row = char_images[0]
    for img in char_images[1:]: row = np.concatenate([row, pad, img], axis=1)
    return row

## 🌐 Step 9: Interactive Web UI (Gradio)

In [ ]:
import gradio as gr

def ui_generate(seed_text, temperature, min_length):
    if not seed_text: seed_text = 'The '
    
    # Generate English text
    generated_chars = generate_text(model, list(str(seed_text)), min_generate=int(min_length), temperature=temperature)
    result_str = seed_text + ''.join(generated_chars)
    
    # Render the FULL generated image (removed the hard cutoff)
    rendered_img = render_handwritten(list(result_str))
    
    return result_str, rendered_img

interface = gr.Interface(
    fn=ui_generate,
    inputs=[
        gr.Textbox(label="Seed Text", placeholder="Type a starting word (e.g. 'King ')...", lines=1),
        gr.Slider(minimum=0.1, maximum=1.5, value=0.5, label="Temperature (Lower = More logical, Higher = More random)"),
        gr.Slider(minimum=10, maximum=100, value=30, step=1, label="Minimum Length (Will naturally finish the last word)")
    ],
    outputs=[gr.Textbox(label="Generated Text"), gr.Image(label="Handwritten Render")],
    title="🖊️ AI Handwritten Text Generator",
    description="This AI was trained on Shakespeare to learn English, and maps its predictions to actual handwritten Kaggle image samples! It will automatically stop at the end of a word so nothing gets cut off."
)

interface.launch(share=True)